# Email Review and Refinement

The workflow adds an email reviewer and an email refiner before sending.

In [ ]:
from dotenv import load_dotenv
import requests
from agents import Agent, Runner, trace, function_tool, ModelSettings
import os
import asyncio
import smtplib
from email.message import EmailMessage

load_dotenv(override=True)
MODEL_NAME = "gpt-5.4-mini"

In [ ]:
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
EMAIL_SMTP_SERVER = os.getenv("EMAIL_SMTP_SERVER")
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

USE_EMAIL = EMAIL_ADDRESS and EMAIL_SMTP_SERVER and EMAIL_APP_PASSWORD

def send_email(subject, text_body, html_body):
    msg = EmailMessage()
    msg["From"] = EMAIL_ADDRESS
    msg["To"] = EMAIL_ADDRESS
    msg["Subject"] = subject
    msg.set_content(text_body)
    msg.add_alternative(html_body, subtype="html")

    with smtplib.SMTP(EMAIL_SMTP_SERVER, 587) as server:
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
        server.send_message(msg)

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

def send_message(subject, text_body, html_body):
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    elif pushover_user and pushover_token:
        push(f"Subject: {subject}\n\n{text_body}")
    else:
        print(f"Subject: {subject}\n\n{text_body}")

In [ ]:
intro = """
You are a sales agent working for ComplAI,
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write emails.
"""

instructions1 = intro + "Your email style is professional, serious, with gravitas and credibility."
instructions2 = intro + "Your email style is witty, engaging, and humorous."
instructions3 = intro + "Your email style is concise, to the point, in the style of a busy senior executive."

sales_agent1 = Agent(name="Professional Sales Agent", instructions=instructions1, model=MODEL_NAME)
sales_agent2 = Agent(name="Humorous Sales Agent", instructions=instructions2, model=MODEL_NAME)
sales_agent3 = Agent(name="Executive Sales Agent", instructions=instructions3, model=MODEL_NAME)

picker_instructions = """
You pick the best cold sales email from the given options.
Imagine you are a customer and pick the one you are most likely to respond to.
Do not give an explanation; reply with the selected email only.
"""

sales_picker = Agent(name="Sales Picker", instructions=picker_instructions, model=MODEL_NAME)

In [ ]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects.

    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    send_message(subject, text_body, html_body)
    return "Email sent successfully"

sender_instructions = """
You receive one final cold sales email.
Use your tool to send that email and only that email.
"""

require_tool = ModelSettings(tool_choice="required")

sales_sender = Agent(
    name="Sales Sender",
    instructions=sender_instructions,
    model=MODEL_NAME,
    tools=[send_email_tool],
    model_settings=require_tool,
)

## Email reviewer and refiner

The reviewer identifies specific improvements without rewriting the draft. The refiner then applies that feedback while retaining the strongest parts of the selected email.

In [ ]:
reviewer_instructions = """
You review cold sales emails for ComplAI.
Assess the email for clarity, credibility, relevance, conciseness, and strength of the call to action.
Give concise, actionable feedback. Do not rewrite the email.
"""

refiner_instructions = """
You refine cold sales emails for ComplAI.
Rewrite the selected email using the reviewer's feedback.
Keep the strongest parts of the original and return only the revised email.
"""

email_reviewer = Agent(name="Email Reviewer", instructions=reviewer_instructions, model=MODEL_NAME)
email_refiner = Agent(name="Email Refiner", instructions=refiner_instructions, model=MODEL_NAME)

## Part 1: Orchestrating the refinement by code

Python fixes the sequence: generate three drafts in parallel, select one, review it, and refine it.

In [ ]:
message = "Write a cold sales email"

with trace("Email refinement by code"):
    draft_results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    draft_outputs = [result.final_output for result in draft_results]

    drafts_for_picker = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(draft_outputs)
    selected = await Runner.run(sales_picker, drafts_for_picker)

    review_request = f"Review this cold sales email:\n\n{selected.final_output}"
    review = await Runner.run(email_reviewer, review_request)

    refinement_request = f"""
Original email:
{selected.final_output}

Reviewer's feedback:
{review.final_output}
"""
    refined = await Runner.run(email_refiner, refinement_request)

print(f"Selected email:\n{selected.final_output}\n")
print(f"Review:\n{review.final_output}\n")
print(f"Refined email:\n{refined.final_output}")

In [ ]:
send_request = f"""
This is the final reviewed email. Send this email and only this email.

{refined.final_output}
"""

with trace("Send refined email by code"):
    send_result = await Runner.run(sales_sender, send_request)

print(send_result.final_output)

## Part 2: Orchestrating the refinement by an LLM

In [ ]:
writer_description = "Write one cold sales email."

sales_writer_tools = [
    sales_agent1.as_tool(tool_name="sales_email_writer_1", tool_description=writer_description),
    sales_agent2.as_tool(tool_name="sales_email_writer_2", tool_description=writer_description),
    sales_agent3.as_tool(tool_name="sales_email_writer_3", tool_description=writer_description),
]

email_review_tool = email_reviewer.as_tool(
    tool_name="email_reviewer",
    tool_description="Review one cold sales email and return concise, actionable feedback.",
)
email_refine_tool = email_refiner.as_tool(
    tool_name="email_refiner",
    tool_description="Refine one email using the reviewer's feedback. Return only the revised email.",
)

refinement_tools = [
    *sales_writer_tools,
    email_review_tool,
    email_refine_tool,
    send_email_tool,
]

refinement_manager_instructions = """
You are a Sales Manager at ComplAI.
Use your team and tools to prepare one reviewed and refined cold sales email.
Follow the requested sequence and send only the final refined email.
"""

refinement_task = """
Follow these steps:

1. Generate Drafts: Use each sales_email_writer tool once to produce three different drafts.
Do not proceed until all three drafts are ready.

2. Evaluate and Select: Compare the drafts and select the single strongest email.

3. Review: Use the email_reviewer tool once to review the selected email.

4. Refine: Use the email_refiner tool once. Give it both the selected email and the reviewer's feedback.

5. Send: Use the send_email_tool to send the refined email. Send one email only.
Do not send an original draft or the reviewer's feedback.
"""

refinement_manager = Agent(
    name="Sales Refinement Manager",
    instructions=refinement_manager_instructions,
    tools=refinement_tools,
    model=MODEL_NAME,
)

In [ ]:
with trace("Sales manager with email refinement"):
    refinement_result = await Runner.run(refinement_manager, refinement_task)

print(refinement_result.final_output)